# Stocker — end-to-end smoke test

Runs the **whole** pipeline (dataset → council → reward → eval → optional GRPO
sanity step → plots) in **under a minute** so you can verify wiring before
the real Colab T4 run.

Stage 1 uses `MockLLMClient` — deterministic, no GPU, no HF download.
Stage 2 (optional) does **one** GRPO step against a tiny model
(`HuggingFaceTB/SmolLM2-135M-Instruct`, 270 MB) to prove the TRL+PEFT
plumbing works. Set `RUN_TINY_GRPO = True` in the last cell to enable it.

In [ ]:
import os, sys

def _is_stocker_root(p: str) -> bool:
    return os.path.isfile(os.path.join(p, 'app', 'council', 'specialists.py'))

for c in (os.getcwd(), '/content/stocker', '/workspace/stocker',
          os.path.expanduser('~/repos/stocker')):
    if _is_stocker_root(c):
        os.chdir(c); sys.path.insert(0, c); WORKDIR = c; break
else:
    raise RuntimeError('No stocker repo on disk in any expected location.')
print('Working dir:', WORKDIR)

In [ ]:
# 1) Verify the bundled dataset is intact. If parquet/charts are missing,
#    rebuild them — but skip if already there (idempotent).
from pathlib import Path
if not (Path('data/prices.parquet').exists() and any(Path('data/charts').glob('*.png'))):
    !python scripts/build_dataset.py
!python scripts/validate_tasks.py

In [ ]:
# 2) Stand up the mock client + a clean council. We disable the on-disk
#    cache (use_cache=False) so the smoke test is independent of any
#    cached Gemma votes from prior real runs.
from app.council.llm import MockLLMClient
from app.council.runner import Council
from app.council.specialists import SPECIALISTS

client = MockLLMClient()
council = Council(client=client, use_cache=False)
print(f'OK: {len(council.specialists)} specialists wired up:',
      [s.name for s in council.specialists])
assert len(council.specialists) == 7

In [ ]:
# 3) Single-step smoke: reset env, run the council once, sanity-check shapes.
import time, json
from app.core.environment import StockerEnv

env = StockerEnv('task_easy')
obs = env.reset().observation
t0 = time.time()
decision = council.run(obs)
dt = time.time() - t0
print(f'Council ran in {dt*1000:.0f} ms')
print(f'  votes:    {len(decision.votes)}')
for v in decision.votes:
    print(f'    {v.name:<18s} signal={v.signal:+.2f}  conf={v.confidence:.2f}')
print(f'  action:    side={decision.action.side}  qty={decision.action.quantity}')
print(f'  rationale: {decision.rationale[:120]}')

assert len(decision.votes) == 7
assert decision.action.side in ('buy', 'sell', 'hold')
assert decision.action.quantity >= 0

In [ ]:
# 4) Full episode of task_easy with mock council — verifies env reward
#    pipeline + done-flag + step counter all hold up across 40+ steps.
env = StockerEnv('task_easy')
obs = env.reset().observation
rewards = []
t0 = time.time()
while True:
    d = council.run(obs)
    r = env.step(d.action)
    rewards.append(r.reward)
    if r.done:
        break
    obs = r.observation
dt = time.time() - t0
print(f'episode steps={len(rewards)}, total_reward={sum(rewards):+.4f}, time={dt:.2f}s')
assert -1.0 <= min(rewards) and max(rewards) <= 1.0, 'reward out of [-1,1]'
assert len(rewards) == env.state().total_steps

In [ ]:
# 5) eval_rollout end-to-end on all 3 tasks — produces reward + portfolio
#    curve PNGs and a summary CSV. Validates the same path GRPO eval uses.
import shutil, glob
shutil.rmtree('training/runs/smoke', ignore_errors=True)
!python -m training.eval_rollout --mock --no-cache --out training/runs/smoke
print()
for f in sorted(glob.glob('training/runs/smoke/*')):
    print(' ', f)

In [ ]:
# 6) Display the curves inline so you see what the real run will produce.
from IPython.display import Image, display
import os
for p in ('training/runs/smoke/reward_curve.png',
         'training/runs/smoke/portfolio_curve.png'):
    if os.path.exists(p):
        print(p)
        display(Image(p))
with open('training/runs/smoke/summary.csv') as f:
    print(f.read())

In [ ]:
# 7) Verify the inference.py CLI emits the expected log markers.
import subprocess
out = subprocess.run(
    ['python', 'inference.py', '--task', 'task_easy', '--mock', '--no-cache'],
    capture_output=True, text=True, timeout=120,
)
lines = out.stdout.splitlines()
counts = {k: sum(1 for ln in lines if ln.startswith(k))
          for k in ('[START]', '[STEP]', '[COUNCIL]', '[END]')}
print('marker counts:', counts)
assert counts['[START]'] == 1
assert counts['[END]'] == 1
assert counts['[STEP]'] >= 10
assert counts['[STEP]'] == counts['[COUNCIL]']
print('OK — log format consistent')

## Stage 2 (optional): real Gemma 4 E4B IT load test

Goes through the **same** `TransformersLLMClient.from_pretrained` path the
real run uses (4-bit BnB, fp16/bf16 auto-detect, `device_map={"":0}`) and
runs a single inference + a single 7-vote council step. Proves:

- the ~12 GB download finishes
- the model fits in your GPU's VRAM at 4-bit
- the chat template + multi-part content (text + image) work end-to-end
  on the *real* processor (not just the mock)

This **is** the loading-flow smoke test you want before kicking off the
full GRPO run. Skipped by default because it needs a GPU and ~12 GB of
disk space; flip `RUN_GEMMA_LOAD = True` in the next cell to enable.

## Stage 3 (optional): tiny-model GRPO sanity

Independent of Stage 2 — confirms TRL + PEFT + LoRA plumbing using a 135M
toy model (270 MB, runs on CPU). Doesn't touch Gemma. Flip
`RUN_TINY_GRPO = True` to enable.

In [ ]:
RUN_GEMMA_LOAD = False    # flip to True to test the actual Gemma loading path

if not RUN_GEMMA_LOAD:
    print("Stage 2 skipped. Set RUN_GEMMA_LOAD = True to test the real load.")
else:
    import torch, time, gc

    print("--- pre-load GPU state ---")
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print(f"  device: {torch.cuda.get_device_name(0)}")
        print(f"  free:   {free / 1024**3:.2f} GiB / {total / 1024**3:.2f} GiB total")
    else:
        print("  no CUDA — Gemma will load on CPU (very slow, will likely OOM)")

    from app.council.llm import TransformersLLMClient

    t0 = time.time()
    gemma = TransformersLLMClient.from_pretrained(
        "google/gemma-4-E4B-it",
        load_in_4bit=True,
    )
    print(f"\nLoaded in {(time.time()-t0)/60:.1f} min")

    # Single text-only inference — confirms chat template + generate work
    print("\n--- generation smoke ---")
    text = gemma.complete(
        [
            {"role": "system", "content": "You are concise. Answer in <= 15 words."},
            {"role": "user",   "content": "Define alpha vs buy-and-hold in stock trading."},
        ],
        max_tokens=80, temperature=0.2,
    )
    print(text)

    # Real council step — exercises the chart-pattern (vision) specialist + moderator
    print("\n--- 7-specialist council vote on real Gemma ---")
    from app.council.runner import Council
    from app.core.environment import StockerEnv
    real_council = Council(client=gemma, use_cache=False)
    obs = StockerEnv("task_easy").reset().observation
    t0 = time.time()
    decision = real_council.run(obs)
    dt = time.time() - t0
    print(f"  {len(decision.votes)} votes in {dt:.1f}s")
    for v in decision.votes:
        print(f"    {v.name:<18s} signal={v.signal:+.2f}  conf={v.confidence:.2f}  "
              f"why='{v.rationale[:80]}'")
    print(f"  action: {decision.action.side}({decision.action.quantity})")
    print(f"  moderator: {decision.rationale[:120]}")

    # Free VRAM in case Stage 3 runs after
    del real_council, gemma
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free, total = torch.cuda.mem_get_info()
        print(f"\nPost-cleanup VRAM: {free / 1024**3:.2f} GiB free")

In [ ]:
RUN_TINY_GRPO = False    # flip to True to verify the GRPO loop end-to-end

if not RUN_TINY_GRPO:
    print('Stage 3 skipped. Set RUN_TINY_GRPO = True to run it.')
else:
    # Colab pre-installs torchao==0.10 which PEFT >=0.13 rejects (needs >=0.16).
    # Upgrade it explicitly here.
    !pip -q install -U 'trl>=0.11' 'peft>=0.13' 'transformers>=4.45' 'accelerate>=1.0' 'torchao>=0.16' datasets

    import importlib.metadata as _m
    print("versions:")
    for pkg in ('trl', 'peft', 'transformers', 'accelerate', 'torchao', 'datasets', 'torch'):
        try:
            print(f"  {pkg:<14s} {_m.version(pkg)}")
        except _m.PackageNotFoundError:
            print(f"  {pkg:<14s} not installed")

    from datasets import Dataset
    from peft import LoraConfig
    from trl import GRPOConfig, GRPOTrainer

    NUM_GEN = 2
    rows = [{'prompt': 'Choose buy/sell/hold and a quantity 1-10. JSON only.'} for _ in range(4)]
    ds = Dataset.from_list(rows)

    def reward_fn(completions, **_):
        return [float(sum(c.isdigit() for c in s)) for s in completions]

    cfg = GRPOConfig(
        output_dir='/tmp/stocker_smoke_grpo',
        per_device_train_batch_size=NUM_GEN,
        gradient_accumulation_steps=1,
        num_generations=NUM_GEN,
        max_steps=1,
        learning_rate=5e-5,
        logging_steps=1,
        report_to=[],
        bf16=False,
    )
    lora = LoraConfig(r=4, lora_alpha=8, target_modules='all-linear', task_type='CAUSAL_LM')
    trainer = GRPOTrainer(
        model='HuggingFaceTB/SmolLM2-135M-Instruct',
        reward_funcs=[reward_fn],
        args=cfg,
        train_dataset=ds,
        peft_config=lora,
    )
    trainer.train()
    print('OK — TRL + PEFT GRPO loop completed 1 step')
    print('  loss/reward log entries:', len(trainer.state.log_history))

## All green?

If every cell above ran without an exception, your stocker pipeline is
wired correctly end-to-end:
- Dataset builds and validates
- Council emits 7 valid votes
- Moderator merges to a legal `(side, quantity)` action
- Env steps reward in `[-1, 1]` and respects `done`
- `eval_rollout` produces curves + CSV
- `inference.py` emits the required log markers
- (optionally) TRL GRPO loop runs to completion

Now you can confidently launch the real Colab T4 run from
`training/train_grpo.ipynb`.